# Parte C · Identificación del gap de conocimiento

**Gap propuesto:** *¿El modelo realmente predice recetas que nunca vio?*

La rúbrica pide argumentar tres cosas, y este notebook las responde con **evidencia cuantitativa** obtenida de los mismos datos y de la reproducción de la Parte B:

| Pregunta de la rúbrica | Sección |
|---|---|
| ¿Qué no respondió el artículo, o qué respondió de forma limitada? | 1 y 2 (evidencias 1–4) |
| ¿Por qué es relevante para la física o la química del problema? | 3 |
| ¿Por qué es viable con el dataset disponible? | 4 |
| Hipótesis que se contrastarán en la Parte D | 5 |
| Relación con la literatura | 6 |

> Aquí **no** se entrena ningún modelo nuevo con partición por formulación: eso es el trabajo de la Parte D. La Parte C solo demuestra que el problema existe, que importa y que se puede resolver con estos datos.

## 1. Qué no respondió el artículo

El paper reporta r² de prueba de hasta 0.98 (Tabla 2) y usa esos modelos para **diseño inverso**: proponer recetas nuevas que den, por ejemplo, partículas de 100 nm. Pero su validación responde a otra pregunta:

- **La partición es por medición, no por receta.** El texto lo dice explícitamente: *"Testing data were randomly subset from the entire database"*. La Fig. S5 del SI se titula *"Number of measurements in either the training data or testing data"*. El código lo confirma: `initial_split(dfML_lipid, prop = 0.85, strata = …)` se aplica sobre las filas (`recursos/02_rf_lipid_model.R`, línea 12).
- **Cada receta se midió 3 veces** (189 formulaciones × 3 = 567 mediciones). Las 3 réplicas tienen **exactamente las mismas X** y valores de y muy parecidos.
- Por lo tanto, casi todas las filas de prueba tienen una "hermana" (otra réplica de la misma receta) en el entrenamiento. Lo que se mide es **qué tan bien se reproduce una medición de una receta conocida**, no **qué tan bien se predice una receta nueva**, que es justamente lo que necesita el diseño inverso.
- Además, el paper no reporta **incertidumbre** de las predicciones ni compara el error del modelo con el **ruido experimental** entre réplicas.

Las siguientes evidencias cuantifican el problema con los datos del autor y con la reproducción de la Parte B.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import r2_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = ROOT / "datos_procesados"
PARTE_B = ROOT / "resultados" / "parte_B"
SALIDA = ROOT / "resultados" / "parte_C"
SALIDA.mkdir(parents=True, exist_ok=True)

SEED, PDI_LIM, TRAIN_PERC, VFOLDS = 1987, 0.250, 0.85, 10
N_SIMULACIONES = 500

C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"          # paleta categórica validada (slots 1-3)
TINTA, TINTA2, REJILLA = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.edgecolor": TINTA2, "axes.labelcolor": TINTA, "xtick.color": TINTA2, "ytick.color": TINTA2,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

ML = {"lipidos": pd.read_csv(DATOS / "ml_lipidos.csv"), "plga": pd.read_csv(DATOS / "ml_plga.csv")}
FAMILIAS = {"lipidos": "Lípidos", "plga": "PLGA"}
OBJETIVOS = {"hdiam_nm": "dH (nm)", "pdi": "PdI", "percEE": "EE (%)"}
MODELOS = [(f, o) for f in FAMILIAS for o in OBJETIVOS]
ETIQUETA = {(f, o): f"{FAMILIAS[f]} · {OBJETIVOS[o]}" for f, o in MODELOS}

PART_B = pd.read_csv(PARTE_B / "particiones_python.csv")
TABLA2_B = pd.read_csv(PARTE_B / "tabla2_rf_python.csv")
PRED_B = pd.read_csv(PARTE_B / "predicciones_rf_python.csv")


def conjunto(familia, objetivo):
    d = ML[familia]
    return d[d["pdi"] <= PDI_LIM] if objetivo == "hdiam_nm" else d


def rsq_cor(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return float(np.corrcoef(y, p)[0, 1] ** 2) if np.std(p) > 1e-12 else np.nan


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))


for (f, o) in MODELOS:
    p = PART_B[(PART_B.familia == f) & (PART_B.objetivo == o)]
    print(f"{ETIQUETA[(f, o)]:16s} train={int((p.conjunto == 'train').sum()):3d} test={int((p.conjunto == 'test').sum()):3d} "
          f"formulaciones={p.meta_formulation_id.nunique()}")

## 2. Evidencia cuantitativa

### Evidencia 1: casi todas las filas de prueba tienen su réplica en entrenamiento

Para cada modelo de la Parte B (misma partición por medición que el paper) se mide:
- el **% de filas de prueba** con al menos una réplica de su misma formulación en el entrenamiento;
- el **% de formulaciones de prueba totalmente nuevas** (ninguna réplica en el entrenamiento);
- lo mismo dentro de la **validación cruzada de 10 folds** que se usa para elegir hiperparámetros.

In [ ]:
def fuga(part):
    tr = part[part.conjunto == "train"]
    te = part[part.conjunto == "test"]
    con_hermana = te.meta_formulation_id.isin(tr.meta_formulation_id)
    form_te = te.meta_formulation_id.unique()
    nuevas = ~pd.Series(form_te).isin(tr.meta_formulation_id)
    # dentro de la CV: filas de validación con réplica en los otros folds
    cv = [tr[tr.fold == k].meta_formulation_id.isin(tr[tr.fold != k].meta_formulation_id).mean()
          for k in sorted(tr.fold.unique())]
    return 100 * con_hermana.mean(), 100 * nuevas.mean(), 100 * np.mean(cv)


filas = []
for (f, o) in MODELOS:
    p = PART_B[(PART_B.familia == f) & (PART_B.objetivo == o)]
    a, b, c = fuga(p)
    filas.append({"Modelo": ETIQUETA[(f, o)], "% filas test con réplica en train": a,
                  "% formulaciones test totalmente nuevas": b, "% filas de validación CV con réplica en otros folds": c})
fuga_b = pd.DataFrame(filas)
fuga_b.to_csv(SALIDA / "evidencia1_fuga_particion_parte_B.csv", index=False)
fuga_b.round(1)

**¿Pasa lo mismo con la partición que genera el código original del autor?** Si ya se ejecutó `00R_referencia_autor.ipynb`, se mide la fuga en las filas exactas que eligió R (`resultados/referencia_R/particiones_R.csv`).

In [ ]:
RUTA_PART_R = ROOT / "resultados" / "referencia_R" / "particiones_R.csv"
if RUTA_PART_R.exists():
    part_r = pd.read_csv(RUTA_PART_R)
    filas = []
    for (f, o) in MODELOS:
        p = part_r[(part_r.familia == f) & (part_r.objetivo == o)].copy()
        p["meta_formulation_id"] = ML[f].loc[p.indice, "meta_formulation_id"].to_numpy()
        a, b, c = fuga(p)
        filas.append({"Modelo": ETIQUETA[(f, o)], "% filas test con réplica en train (partición R)": a,
                      "% formulaciones test totalmente nuevas (partición R)": b,
                      "% validación CV con réplica en otros folds (partición R)": c})
    fuga_r = pd.DataFrame(filas)
    fuga_r.to_csv(SALIDA / "evidencia1_fuga_particion_R_autor.csv", index=False)
    display(fuga_r.round(1))
else:
    print("No existe resultados/referencia_R/particiones_R.csv → ejecutar 00R_referencia_autor.ipynb.")

¿Es casualidad de esta semilla? Se repite la **misma partición del autor** (85/15 por medición, estratificada por cuartiles del objetivo) con **500 semillas distintas**. Las funciones son idénticas a las de la Parte B.

In [ ]:
def estratos_rsample(y, cortes=4, pool=0.10):
    e = np.asarray(pd.qcut(y, q=cortes, labels=False, duplicates="drop"))
    while True:
        etiquetas, n = np.unique(e, return_counts=True)
        if len(etiquetas) <= 1 or n.min() >= pool * len(y):
            return e
        i = int(np.argmin(n))
        e[e == etiquetas[i]] = etiquetas[i - 1] if i > 0 else etiquetas[i + 1]


def particion_por_medicion(d, objetivo, semilla):
    y = d[objetivo].to_numpy()
    rng = np.random.default_rng(semilla)
    estratos = estratos_rsample(y)
    en_train = np.zeros(len(y), dtype=bool)
    for e in np.unique(estratos):
        idx = np.flatnonzero(estratos == e)
        en_train[rng.choice(idx, int(np.floor(len(idx) * TRAIN_PERC)), replace=False)] = True
    return en_train


def memorizador(d, objetivo, en_train):
    # Predice cada fila de prueba con el promedio de las réplicas de su formulación que están en train.
    # Si la formulación no tiene réplicas en train, usa la media global de train.
    media_form = d[en_train].groupby("meta_formulation_id")[objetivo].mean()
    te = d[~en_train]
    pred = te.meta_formulation_id.map(media_form)
    vista = pred.notna().to_numpy()
    return te[objetivo].to_numpy(), pred.fillna(d.loc[en_train, objetivo].mean()).to_numpy(), vista


sim = []
for (f, o) in MODELOS:
    d = conjunto(f, o)
    for s in range(N_SIMULACIONES):
        en_train = particion_por_medicion(d, o, s)
        y, p, vista = memorizador(d, o, en_train)
        sim.append({"familia": f, "objetivo": o, "semilla": s, "pct_hermana": 100 * vista.mean(),
                    "mem_rsq": rsq_cor(y, p), "mem_rmse": rmse(y, p),
                    "mem_rsq_vistas": rsq_cor(y[vista], p[vista]), "mem_rmse_vistas": rmse(y[vista], p[vista])})
sim = pd.DataFrame(sim)
sim.to_csv(SALIDA / "simulacion_500_particiones.csv", index=False)

resumen_sim = sim.groupby(["familia", "objetivo"]).agg(
    pct_hermana_mediana=("pct_hermana", "median"), pct_hermana_min=("pct_hermana", "min"),
    pct_hermana_max=("pct_hermana", "max")).reset_index()
resumen_sim["Modelo"] = [ETIQUETA[(f, o)] for f, o in zip(resumen_sim.familia, resumen_sim.objetivo)]
resumen_sim[["Modelo", "pct_hermana_mediana", "pct_hermana_min", "pct_hermana_max"]].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
etiquetas = [ETIQUETA[m] for m in MODELOS]
datos = [sim[(sim.familia == f) & (sim.objetivo == o)].pct_hermana.to_numpy() for f, o in MODELOS]
pos = np.arange(len(MODELOS))
partes = ax.violinplot(datos, positions=pos, showextrema=False, widths=0.75)
for cuerpo in partes["bodies"]:
    cuerpo.set_facecolor(C1); cuerpo.set_edgecolor(C1); cuerpo.set_alpha(0.35)
ax.scatter(pos, fuga_b["% filas test con réplica en train"], color=C2, s=60, zorder=3,
           edgecolor="white", linewidth=0.8, label="partición de la Parte B (semilla 1987)")
ax.scatter([], [], color=C1, alpha=0.5, s=60, marker="s", label="500 particiones simuladas")
ax.set_xticks(pos, etiquetas, rotation=20, ha="right")
ax.set_ylabel("% de filas de prueba con\nsu réplica en entrenamiento")
ax.set_ylim(np.floor(min(np.concatenate(datos).min(), fuga_b["% filas test con réplica en train"].min())) - 2, 100.5)
ax.legend(loc="lower left")
ax.set_title("Evidencia 1 · La partición por medición casi nunca deja recetas nuevas en la prueba", color=TINTA)
fig.tight_layout()
fig.savefig(SALIDA / "fig_evidencia1_fuga.png", bbox_inches="tight")
plt.show()

### Evidencia 2: un "modelo" que no aprende nada alcanza los números del paper

El **memorizador** no es un modelo de ML. Para cada fila de prueba, devuelve **el promedio de las réplicas de su misma receta que quedaron en el entrenamiento**; si no hay ninguna, devuelve el promedio global.

Si la partición del paper midiera la capacidad de predecir recetas nuevas, este predictor trivial debería quedar muy por debajo del Random Forest. Si queda **al mismo nivel**, el protocolo de validación **no distingue** entre "aprendió la relación receta → propiedad" y "recordó la receta".

In [ ]:
TABLA2_PAPER = {("lipidos", "hdiam_nm"): (9.5, 0.922), ("lipidos", "pdi"): (0.053, 0.787),
                ("lipidos", "percEE"): (10.6, 0.472), ("plga", "hdiam_nm"): (7.5, 0.964),
                ("plga", "pdi"): (0.047, 0.560), ("plga", "percEE"): (2.2, 0.984)}
filas = []
for (f, o) in MODELOS:
    d = conjunto(f, o)
    p = PART_B[(PART_B.familia == f) & (PART_B.objetivo == o)].set_index("indice")
    en_train = (p.loc[d.index, "conjunto"] == "train").to_numpy()
    y, pm, vista = memorizador(d, o, en_train)
    # predicciones del RF de la Parte B en las mismas filas de prueba
    rf = PRED_B[(PRED_B.familia == f) & (PRED_B.objetivo == o) & (PRED_B.conjunto == "test")].set_index("indice")
    p_rf = rf.loc[d.index[~en_train], "pred"].to_numpy()
    s = sim[(sim.familia == f) & (sim.objetivo == o)]
    filas.append({
        "Modelo": ETIQUETA[(f, o)],
        "r² memorizador (Parte B)": rsq_cor(y, pm), "r² RF Python (Parte B)": rsq_cor(y, p_rf),
        "r² RF paper (Tabla 2)": TABLA2_PAPER[(f, o)][1],
        "r² memorizador (mediana 500)": s.mem_rsq.median(),
        "RMSE memorizador (Parte B)": rmse(y, pm), "RMSE RF Python (Parte B)": rmse(y, p_rf),
        "RMSE RF paper (Tabla 2)": TABLA2_PAPER[(f, o)][0],
        "corr(pred RF, pred memorizador)": float(np.corrcoef(p_rf, pm)[0, 1]),
    })
memo = pd.DataFrame(filas)
memo.to_csv(SALIDA / "evidencia2_memorizador.csv", index=False)
memo.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
x = np.arange(len(memo)); w = 0.26
series = [("r² memorizador (Parte B)", C1, "memorizador (sin aprendizaje)"),
          ("r² RF Python (Parte B)", C2, "Random Forest · reproducción Python"),
          ("r² RF paper (Tabla 2)", C3, "Random Forest · paper (Tabla 2)")]
for k, (col, color, nombre) in enumerate(series):
    ax.bar(x + (k - 1) * w, memo[col], width=w - 0.03, color=color, label=nombre, zorder=2)
ax.set_xticks(x, memo["Modelo"], rotation=20, ha="right")
ax.set_ylabel("r² en el conjunto de prueba")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=3)
ax.set_title("Evidencia 2 · Recordar la receta rinde casi lo mismo que el Random Forest", color=TINTA)
fig.tight_layout()
fig.savefig(SALIDA / "fig_evidencia2_memorizador.png", bbox_inches="tight")
plt.show()

#### ¿Y cuando la receta de prueba sí es nueva?

La partición por medición deja muy pocas filas de prueba de recetas nunca vistas (Evidencia 1), pero en algunos modelos hay unas cuantas. Separar el error **del mismo Random Forest de la Parte B** según si la fila de prueba tiene o no una réplica en entrenamiento da un adelanto directo del gap. No se entrena ningún modelo nuevo.

> ⚠️ Son muy pocas filas (0–6 por modelo). Es un indicio, no una prueba. La medición rigurosa, con partición por receta, es la Parte D (H1).

In [ ]:
filas = []
for (f, o) in MODELOS:
    d = conjunto(f, o)
    p = PART_B[(PART_B.familia == f) & (PART_B.objetivo == o)].set_index("indice")
    en_train = (p.loc[d.index, "conjunto"] == "train").to_numpy()
    te = d[~en_train]
    vista = te.meta_formulation_id.isin(d[en_train].meta_formulation_id).to_numpy()
    rf = PRED_B[(PRED_B.familia == f) & (PRED_B.objetivo == o) & (PRED_B.conjunto == "test")].set_index("indice")
    err = te[o].to_numpy() - rf.loc[te.index, "pred"].to_numpy()
    filas.append({"Modelo": ETIQUETA[(f, o)],
                  "filas test con réplica en train": int(vista.sum()),
                  "RMSE RF (receta ya vista)": float(np.sqrt(np.mean(err[vista] ** 2))),
                  "filas test de receta nueva": int((~vista).sum()),
                  "RMSE RF (receta nueva)": float(np.sqrt(np.mean(err[~vista] ** 2))) if (~vista).any() else np.nan})
vista_vs_nueva = pd.DataFrame(filas)
vista_vs_nueva["cociente nueva / vista"] = vista_vs_nueva["RMSE RF (receta nueva)"] / vista_vs_nueva["RMSE RF (receta ya vista)"]
vista_vs_nueva.to_csv(SALIDA / "evidencia2b_error_receta_vista_vs_nueva.csv", index=False)
vista_vs_nueva.round(3)

### Evidencia 3: el error del modelo frente al ruido experimental

Las 3 réplicas de una receta no dan exactamente el mismo valor: hay variabilidad de fabricación y de medición (DLS, UV-Vis). Ese **ruido entre réplicas** es un piso de error que ningún modelo puede superar al predecir una medición nueva.

- **SD agrupada**: √(Σ residuos respecto a la media de su receta² / (n − n.º de recetas)). Es comparable con un RMSE.
- **Techo de r²**: la fracción de la variación total que se explica con la media de cada receta. Ningún modelo que use solo la receta puede superarlo.

In [ ]:
filas = []
for (f, o) in MODELOS:
    d = conjunto(f, o)
    g = d.groupby("meta_formulation_id")[o]
    resid = d[o] - g.transform("mean")
    sd_agrupada = float(np.sqrt((resid ** 2).sum() / (len(d) - d.meta_formulation_id.nunique())))
    techo = 1 - (resid ** 2).sum() / ((d[o] - d[o].mean()) ** 2).sum()
    rf_b = TABLA2_B[(TABLA2_B.Familia == FAMILIAS[f]) & (TABLA2_B.Objetivo == OBJETIVOS[o])].iloc[0]
    filas.append({"Modelo": ETIQUETA[(f, o)], "SD entre réplicas (agrupada)": sd_agrupada,
                  "SD entre réplicas (mediana)": float(g.std().median()), "Techo de r²": techo,
                  "RMSE test RF Python": rf_b.rmse_test, "RMSE test RF paper": TABLA2_PAPER[(f, o)][0],
                  "RMSE paper / SD agrupada": TABLA2_PAPER[(f, o)][0] / sd_agrupada})
ruido = pd.DataFrame(filas)
ruido.to_csv(SALIDA / "evidencia3_ruido_replicas.csv", index=False)
ruido.round(3)

### Evidencia 4: cómo se ve la diferencia entre partir por medición y partir por receta

Cada fila del gráfico es una receta (formulación) y cada cuadro una de sus 3 réplicas. A la izquierda, la partición que usa el paper (y la Parte B) para el modelo de PdI de PLGA. A la derecha, una partición **por receta** (las 3 réplicas siempre juntas), que es la que se propone para la Parte D.

In [ ]:
f, o = "plga", "pdi"
d = conjunto(f, o)
p = PART_B[(PART_B.familia == f) & (PART_B.objetivo == o)].set_index("indice")
d = d.assign(conj_med=p.loc[d.index, "conjunto"].to_numpy())
# 14 recetas con al menos una réplica en prueba (para que se vea el efecto)
recetas = d.loc[d.conj_med == "test", "meta_formulation_id"].unique()[:14]
sub = d[d.meta_formulation_id.isin(recetas)].copy()
sub["rep"] = sub.groupby("meta_formulation_id").cumcount()
# partición por receta: ~15 % de las recetas completas a prueba
rng = np.random.default_rng(SEED)
prueba_por_receta = set(rng.choice(recetas, size=max(2, int(round(0.15 * len(recetas)))), replace=False))

fig, ejes = plt.subplots(1, 2, figsize=(9, 5.2), sharey=True)
for ax, titulo, regla in [(ejes[0], "Paper: partición por medición", lambda r: r.conj_med == "test"),
                          (ejes[1], "Propuesta: partición por receta", lambda r: r.meta_formulation_id in prueba_por_receta)]:
    for i, rec in enumerate(recetas):
        for _, r in sub[sub.meta_formulation_id == rec].iterrows():
            color = C2 if regla(r) else C1
            ax.add_patch(plt.Rectangle((r.rep + 0.06, i + 0.06), 0.88, 0.88, color=color, linewidth=0))
    ax.set_xlim(0, 3); ax.set_ylim(len(recetas), 0)
    ax.set_xticks([0.5, 1.5, 2.5], ["réplica 1", "réplica 2", "réplica 3"])
    ax.set_yticks(np.arange(len(recetas)) + 0.5, recetas)
    ax.grid(False)
    ax.set_title(titulo, color=TINTA)
ejes[0].set_ylabel("Receta (formulación)")
manijas = [plt.Rectangle((0, 0), 1, 1, color=C1), plt.Rectangle((0, 0), 1, 1, color=C2)]
fig.legend(manijas, ["entrenamiento", "prueba"], loc="lower center", ncol=2)
fig.suptitle("Evidencia 4 · Con partición por medición, las recetas de prueba ya se vieron en el entrenamiento",
             color=TINTA, fontsize=11)
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(SALIDA / "fig_evidencia4_esquema_particion.png", bbox_inches="tight")
plt.show()

### Síntesis de la evidencia

| Evidencia | Resultado | Qué significa |
|---|---|---|
| 1. Fuga en la partición | Entre el **83 % y el 100 %** de las filas de prueba tienen su réplica en entrenamiento, tanto en la partición de la Parte B como en **la partición real que genera el código del autor en R** (83–100 %). En 500 particiones simuladas, la mediana es **92–100 %**. En la CV de 10 folds, el **85–95 %** | La prueba del paper casi no contiene recetas nuevas, y no depende de la implementación |
| 2. Memorizador | Sin aprender nada, alcanza r² de prueba de **0.72–0.96**, al nivel del RF reproducido (0.70–0.99) y del paper. Las predicciones del RF y del memorizador tienen correlación de **0.96–0.997** | El protocolo no distingue "aprendió la física" de "recordó la receta". En la práctica, el RF se comporta como una tabla de consulta de réplicas |
| 2b. Receta vista vs nueva | Solo 2 de los 6 modelos tienen filas de recetas nuevas en la prueba. **PLGA · dH:** RMSE de 5.5 nm en recetas vistas frente a **17.4 nm** en las 6 filas nuevas (3.2 veces más). **PLGA · EE:** lo contrario, 3.5 % frente a **1.2 %** en las 3 filas nuevas | Resultado **mixto y con muy pocas filas**: no permite concluir, pero muestra que la validación del paper no puede responder la pregunta. El contraste en EE de PLGA es coherente con H2: la EE la determina sobre todo la proporción fármaco:polímero, que también aparece en otras recetas |
| 3. Ruido entre réplicas | El RMSE de prueba del paper es **0.7–1.6 veces** la SD entre réplicas. En dH de lípidos y EE de PLGA queda **por debajo** del propio ruido experimental | Un error por debajo del ruido de medición, al predecir mediciones nuevas, es otra señal de que la prueba no es independiente del entrenamiento |
| 4. Esquema | — | Ilustra la diferencia entre partir por medición y partir por receta |

## 3. Por qué es relevante para la física y la química del problema

1. **El uso real del modelo es predecir recetas nuevas.** El objetivo del paper es el **diseño inverso**: dado un tamaño deseado, proponer TFR, FRR y concentraciones. Cada receta propuesta es, por definición, una combinación que no está en la base. Un error medido sobre recetas ya vistas no dice cuánto confiar en esa propuesta, ni cuántos experimentos de laboratorio se ahorran realmente.
2. **Las réplicas separan dos fuentes de variación distintas:**
   - **entre recetas**, que es la física de la nanoprecipitación: velocidad de mezcla (TFR), dilución del solvente (FRR), sobresaturación (concentración);
   - **dentro de una receta**, que es ruido de fabricación y de medición (DLS, UV-Vis).

   Un modelo útil debe capturar la primera. Con la partición por medición, el modelo puede rendir bien solo con reconocer la receta, sin haber aprendido ninguna relación física.
3. **El PdI es un caso especialmente sensible.** Su RMSE en el paper (0.053 en lípidos y 0.047 en PLGA) ya está **al nivel del ruido entre réplicas** (Evidencia 3). Esto es coherente con un problema físico conocido: en muestras polidispersas, el ajuste de cumulantes de la DLS es poco confiable. Saber si el modelo supera ese ruido en recetas nuevas define si sirve para controlar la uniformidad, que es un requisito de calidad de una nanomedicina.
4. **El diseño experimental es una rejilla de 3 niveles por variable.** Predecir una receta nueva dentro de la rejilla es **interpolar** a una combinación no vista de niveles conocidos. Predecir fuera de ella (un TFR o una concentración nuevos) es **extrapolar**. Las dos tareas tienen sentido físico distinto y el paper no evalúa ninguna de las dos por separado.
5. **La interpretación física (SHAP) depende de que el modelo haya aprendido la relación correcta.** Si el modelo memoriza, las importancias pueden reflejar qué columnas identifican la receta y no qué variables controlan la formación de la partícula. Por ejemplo, el colesterol, colineal con la concentración total, aparece como "lo más importante".

## 4. Por qué es viable con el dataset disponible

- **El agrupamiento se puede reconstruir sin información externa:** las réplicas tienen X idénticas, y el notebook `00` obtiene exactamente 81 + 108 = 189 formulaciones de 3 réplicas.
- **Hay suficientes grupos** para una validación agrupada. Abajo se comprueba que una partición GroupKFold de 10 folds deja entre 5 y 11 recetas completas por fold, sin romper ninguna receta.
- **Solo requiere ML clásico** (Random Forest, kNN, procesos gaussianos, Quantile Random Forest o bootstrap para la incertidumbre) y las mismas variables del paper, como piden las reglas de la Parte D.
- **La línea base ya existe:** la reproducción de la Parte B, con la partición del paper, es el punto de comparación directo.

In [ ]:
filas = []
for (f, o) in MODELOS:
    d = conjunto(f, o)
    gkf = GroupKFold(n_splits=VFOLDS)
    recetas_por_fold, rotas = [], 0
    for tr, va in gkf.split(d, groups=d.meta_formulation_id):
        recetas_por_fold.append(d.iloc[va].meta_formulation_id.nunique())
        rotas += len(set(d.iloc[va].meta_formulation_id) & set(d.iloc[tr].meta_formulation_id))
    filas.append({"Modelo": ETIQUETA[(f, o)], "filas": len(d), "recetas": d.meta_formulation_id.nunique(),
                  "recetas por fold (mín–máx)": f"{min(recetas_por_fold)}–{max(recetas_por_fold)}",
                  "recetas repartidas entre train y validación": rotas})
viabilidad = pd.DataFrame(filas)
viabilidad.to_csv(SALIDA / "viabilidad_groupkfold.csv", index=False)
viabilidad

## 5. Hipótesis para la Parte D (formuladas antes de los experimentos)

> **H1. Generalización a recetas nuevas.** Si se reemplaza la partición por medición por una **partición por receta** (GroupKFold / leave-one-formulation-out), el r² de prueba del Random Forest **bajará** respecto al de la reproducción (Parte B), sobre todo en **PdI** y en **dH de lípidos**. El RMSE de prueba aumentará, como mínimo, hasta superar el ruido entre réplicas.
>
> **H2. Error frente al ruido experimental.** Con la partición por receta:
> - el error del RF en **dH y EE de PLGA** se mantendrá en el mismo orden que el ruido entre réplicas, porque son propiedades bien determinadas por la composición: la concentración de polímero y la proporción fármaco:polímero;
> - en **PdI**, el error quedará claramente por encima de ese ruido.
>
> **H3. Incertidumbre.** Un método clásico de incertidumbre (Quantile Random Forest o bootstrap) evaluado con partición por receta producirá intervalos con **cobertura cercana a la nominal** en dH de PLGA, y **más anchos** en PdI y en los modelos de lípidos.

**Criterios de contraste** (se reportarán en la Parte D):
- diferencia de RMSE y r² entre particiones, con su dispersión entre folds o repeticiones;
- cociente RMSE / SD entre réplicas;
- cobertura y ancho medio de los intervalos.

## 6. Relación con la literatura

- **Fuga de información (*data leakage*).** Kapoor y Narayanan (2023) documentan que la fuga afecta a cientos de publicaciones de ML en ciencia y que, al corregirla, los modelos complejos pierden su ventaja aparente. En su taxonomía, que las muestras de prueba no sean independientes de las de entrenamiento es una forma de fuga. Las pautas REFORMS (Kapoor *et al.*, 2023) piden explícitamente que las **muestras dependientes**, como las réplicas, queden juntas en la misma partición.
- **Validación con datos estructurados.** Roberts *et al.* (2017) muestran con simulaciones y casos reales que la validación cruzada aleatoria **subestima el error de predicción** cuando los datos tienen estructura (grupos, jerarquías). Recomiendan la **validación por bloques o grupos** cuando el objetivo es predecir datos nuevos.
- **Descubrimiento de materiales.** Meredig *et al.* (2018) proponen la validación *leave-one-cluster-out* y comparar contra un **predictor trivial de vecino más cercano**: las métricas tradicionales sobreestiman el desempeño de los modelos cuando se usan para proponer materiales nuevos. El "memorizador" de la Evidencia 2 sigue esa misma lógica.
- **Partición para evitar fugas.** Joeres *et al.* (2025, DataSAIL) formalizan la partición de datos para estimar el desempeño fuera de la distribución de entrenamiento.
- **ML en formulación microfluídica.** El trabajo previo del mismo grupo (Di Francesco *et al.*, 2023), con liposomas cargados de curcumina, y otros estudios de ML para el tamaño de partículas de PLGA en microfluídica (Damiati *et al.*, 2020) reportan métricas altas, pero no evalúan explícitamente la predicción sobre recetas nunca vistas. Esa es la contribución que se propone.

### Referencias

1. Moore, T. L.; Pesce, C.; Greco, A.; Pisante, C.; Avancini, G.; Di Francesco, V.; Shamay, Y.; Decuzzi, P. Unleashing the Power of Machine Learning in Nanomedicine Formulation Development. *Adv. Funct. Mater.* **2026**, *36*, e14387. https://doi.org/10.1002/adfm.202514387
2. Kapoor, S.; Narayanan, A. Leakage and the reproducibility crisis in machine-learning-based science. *Patterns* **2023**, *4*, 100804. https://doi.org/10.1016/j.patter.2023.100804
3. Kapoor, S. *et al.* REFORMS: Reporting Standards for Machine Learning Based Science. *arXiv* **2023**, 2308.07832. https://arxiv.org/abs/2308.07832
4. Roberts, D. R. *et al.* Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography* **2017**, *40*, 913–929. https://doi.org/10.1111/ecog.02881
5. Meredig, B. *et al.* Can machine learning identify the next high-temperature superconductor? Examining extrapolation performance for materials discovery. *Mol. Syst. Des. Eng.* **2018**, *3*, 819–825. https://doi.org/10.1039/C8ME00012C
6. Joeres, R.; Blumenthal, D. B.; Kalinina, O. V. Data splitting to avoid information leakage with DataSAIL. *Nat. Commun.* **2025**, *16*, 3337. https://doi.org/10.1038/s41467-025-58606-8
7. Di Francesco, V. *et al.* Machine learning instructed microfluidic synthesis of curcumin-loaded liposomes. *Biomed. Microdevices* **2023**, *25*, 29. https://doi.org/10.1007/s10544-023-00671-1
8. Damiati, S. A. *et al.* Artificial intelligence application for rapid fabrication of size-tunable PLGA microparticles in microfluidics. *Sci. Rep.* **2020**, *10*, 19517. https://doi.org/10.1038/s41598-020-76477-5